In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import asyncio
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["pandas", "scikit-learn"]
ASSETS = [{"name": "fitness analysis.csv", "url": "/labs/2.2.3/fitness%20analysis.csv", "size": 166379, "sha256": "82427ce1064ecb1777e371b122ff15a16c941a66ecc534fb4e9a9f6cce16dff4"}]

async def _retry(label, operation, attempts=3):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return await operation()
        except Exception as error:
            last_error = error
            if attempt < attempts:
                await asyncio.sleep(0.8 * attempt)
    raise RuntimeError(f"{label}失败（已重试 {attempts} 次）：{last_error}") from last_error

if PACKAGES:
    await _retry("依赖安装", lambda: piplite.install(PACKAGES))

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    async def _download():
        separator = "&" if "?" in asset["url"] else "?"
        response = await pyfetch(f'{asset["url"]}{separator}sha256={asset["sha256"][:16]}')
        if response.status < 200 or response.status >= 300:
            raise RuntimeError(f'HTTP {response.status}：{asset["url"]}')
        return await response.bytes()
    payload = await _retry(f'素材下载 {asset["name"]}', _download)
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 2.2.3 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import pickle
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

# 加载数据集
df = __________

# 显示前五行数据
print(__________)

# 去除所有字符串字段的前后空格
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# 检查和清理列名
df.columns = df.columns.str.strip()

# 选择相关特征进行建模
X = df[['Your gender', 'How important is exercise to you ?', 'How healthy do you consider yourself?']]
X = __________(X)  # 将分类变量转为数值变量

# 将年龄段转为数值变量
y = __________(lambda x: int(x.split(' ')[0]))  # 假设年龄段为整数

# 将数据集划分为训练集和测试集（测试集占比20%）
X_train, X_test, y_train, y_test = __________(__________, random_state=42)

# 创建随机森林回归模型（创建的决策树的数量为100）
rf_model = __________(__________, random_state=42)
# 训练随机森林回归模型
__________

# 保存训练好的模型
with open('2.2.3_model.pkl', 'wb') as model_file:
    pickle.__________

# 进行结果预测
y_pred = __________
results_df = pd.DataFrame(y_pred, columns=['预测结果'])
results_df.to_csv('2.2.3_results.txt', index=False)

# 使用测试工具对模型进行测试，并记录测试结果
train_score = __________   #训练集分数
test_score = __________    #测试集分数
mse = __________  #均方误差
r2 = __________  #决定系数
with open('2.2.3_report.txt', 'w') as report_file:
    report_file.write(f'训练集得分: {train_score}\n')
    report_file.write(f'测试集得分: {test_score}\n')
    report_file.write(f'均方误差(MSE): {mse}\n')
    report_file.write(f'决定系数(R^2): {r2}\n')

# 运用工具分析算法中错误案例产生的原因并进行纠正
# 初始化XGBoost回归模型（构建100棵树）
xgb_model = __________(__________, random_state=42)
# 训练XGBoost回归模型
__________
# 使用XGBoost回归模型在测试集上进行结果预测
y_pred_xgb = __________

results_df_xgb = pd.DataFrame(y_pred_xgb, columns=['预测结果'])
results_df_xgb.to_csv('2.2.3_results_xgb.txt', index=False)

with open('2.2.3_report_xgb.txt', 'w') as xgb_report_file:
    xgb_report_file.write(f'XGBoost训练集得分: {__________}\n')
    xgb_report_file.write(f'XGBoost测试集得分: {__________}\n')
    xgb_report_file.write(f'XGBoost均方误差(MSE): {__________}\n')
    xgb_report_file.write(f'XGBoost决定系数(R^2): {__________)}\n')
